# ARTEMIS — LoRA Fine-tuning (Colab GPU)

Requiere **GPU en Colab** (A100 recomendado ~2.5 h, T4 ~5-7 h).

**Prerequisito**: haber corrido `rag.ipynb` en local y subido a Drive:
- `train_processed.pkl`
- `val_processed.pkl`
- `test_processed.pkl`
- `retrieval_index.json`

Produce:
- `decoder_checkpoint/` — checkpoint LoRA (entregable obligatorio)
- `submission.csv` — predicciones sobre test.csv

## 0 · Setup Colab

In [1]:
# Montar Drive
from google.colab import drive
drive.mount("/content/drive")

# Ajusta este path al lugar donde subiste los archivos
DRIVE_PATH = "/content/drive/MyDrive/MASTER/Tercer_semestre/NLP_2/Competencia"

Mounted at /content/drive


In [2]:
# Pinear versiones compatibles entre sí — IMPORTANTE: reiniciar el runtime después
!pip install -q \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "trl==0.12.2" \
    "accelerate==1.1.1" \
    "datasets==3.1.0"

print("✓ Instalación OK — REINICIA EL RUNTIME antes de continuar (Entorno de ejecución → Reiniciar)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 149.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 123.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.

In [3]:
import json
import pickle
import re
from difflib import get_close_matches
from pathlib import Path

import numpy as np
import pandas as pd
import torch

BASE = Path(DRIVE_PATH)

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — revisar runtime'}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

GPU: NVIDIA A100-SXM4-40GB
CUDA disponible: True


## 1 · Cargar datos pre-procesados

In [4]:
train_df = pd.read_pickle(BASE / "train_processed.pkl")
val_df   = pd.read_pickle(BASE / "val_processed.pkl")
test_df  = pd.read_pickle(BASE / "test_processed.pkl")

with open(BASE / "Data" / "tools_definition.json") as f:
    tools_def_raw = json.load(f)
TOOLS_DEF   = {t["name"]: t for t in tools_def_raw["tools"]}
VALID_TOOLS = list(TOOLS_DEF.keys())

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Herramientas: {VALID_TOOLS}")

Train: 2313 | Val: 257 | Test: 766
Herramientas: ['get_telemetry', 'get_crew_status', 'get_module_status', 'send_alert', 'send_message', 'schedule_maintenance', 'activate_protocol', 'control_system', 'calculate_trajectory', 'request_supply', 'no_action']


## 2 · Cargar modelo base + LoRA

In [ ]:
from huggingface_hub import login

# Obtén tu token en https://huggingface.co/settings/tokens (tipo "Read")
# Acepta la licencia de Llama en https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
login(token="hf_TU_TOKEN_AQUI")

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_cfg = LoraConfig(
    r=64,
    lora_alpha=128,          # 2×r
    lora_dropout=0.05,
    target_modules=[         # attention + MLP completo
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
# Esperado: ~45M trainable / 1.24B total (3.6%)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 45,088,768 || all params: 1,280,903,168 || trainable%: 3.5201


## 3 · Dataset de entrenamiento

Solo se entrena el `tool_call` — el contexto (query + chunks recuperados) se enmascara con `labels = -100`.  
Mismo patrón que Microproyecto 3.

In [7]:
SYSTEM_PROMPT = """You are ARTEMIS, the AI control system for MASA's Kuntur Station.
Given an operator query and relevant documentation, output ONLY the exact tool call.

Format rules (STRICT — any deviation = wrong answer):
- No spaces after commas or around '=' signs
- Single quotes for string values: module='condor'
- Integer values without quotes: timeframe_hours=6
- Parameter ORDER must match the tool definition exactly
- Module names are lowercase ASCII: condor, quetzal, jaguar, colibri, vicuna, tucan
- Protocol IDs UPPERCASE: MASA-SEC-012
- For purely informational queries with no system action: no_action"""

def make_prompt(query: str, ctx_chunks: list) -> str:
    ctx = "\n\n".join(f"[{c['doc_id']}]\n{c['text']}" for c in ctx_chunks)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Query: {query}\n\nContext:\n{ctx}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [8]:
from torch.utils.data import Dataset as TorchDataset

MAX_SEQ      = 2048   # A100-40GB aguanta batch=4, seq=2048 para Llama-1B
LABEL_IGNORE = -100
PROMPT_K     = 3      # PKLs tienen 3 chunks; chunks de ~400 tok → prompt ~1400 tok total

class ToolCallDataset(TorchDataset):
    def __init__(self, df: pd.DataFrame):
        self.examples = []
        skipped = 0
        for _, row in df.iterrows():
            prompt   = make_prompt(row["query"], row["context_chunks"][:PROMPT_K])
            full_str = prompt + row["tool_call"] + tokenizer.eos_token

            prompt_ids = tokenizer.encode(prompt,    add_special_tokens=False)
            full_ids   = tokenizer.encode(full_str,  add_special_tokens=False)

            if len(full_ids) > MAX_SEQ:
                full_ids   = full_ids[:MAX_SEQ]
                prompt_ids = prompt_ids[:MAX_SEQ]

            n_ctx  = min(len(prompt_ids), len(full_ids))
            labels = [LABEL_IGNORE] * n_ctx + full_ids[n_ctx:]

            if len(labels) > n_ctx:
                self.examples.append({
                    "input_ids":      full_ids,
                    "attention_mask": [1] * len(full_ids),
                    "labels":         labels,
                })
            else:
                skipped += 1

        total = len(self.examples) + skipped
        print(f"  {len(self.examples)}/{total} ejemplos usados | {skipped} descartados ({skipped/total*100:.0f}%)")
        if skipped / max(total, 1) > 0.10:
            print(f"  AVISO: >10% descartados — aumenta MAX_SEQ")

    def __len__(self):         return len(self.examples)
    def __getitem__(self, i):  return self.examples[i]


print("Construyendo train dataset...")
train_dataset = ToolCallDataset(train_df)
print("Construyendo val dataset...")
val_dataset   = ToolCallDataset(val_df)

Construyendo train dataset...
  2313/2313 ejemplos usados | 0 descartados (0%)
Construyendo val dataset...
  257/257 ejemplos usados | 0 descartados (0%)


## 4 · Entrenamiento

In [9]:
from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

CKPT_DIR = str(BASE / "decoder_checkpoint")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=LABEL_IGNORE,
    pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,   # batch efectivo = 32
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    dataloader_num_workers=2,
    report_to="none",
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

Epoch,Training Loss,Validation Loss
0,0.112100,0.056971
1,0.038400,0.038184
2,0.022100,0.023423
3,0.011000,0.015650
4,0.006700,0.015484
5,0.003900,0.016535
6,0.002300,0.017834
7,0.002000,0.017441


TrainOutput(global_step=576, training_loss=0.06375620332093807, metrics={'train_runtime': 1195.9276, 'train_samples_per_second': 15.473, 'train_steps_per_second': 0.482, 'total_flos': 9.285676502089728e+16, 'train_loss': 0.06375620332093807, 'epoch': 7.982728842832469})

In [10]:
# Guardar checkpoint final (entregable obligatorio)
model.save_pretrained(CKPT_DIR)
tokenizer.save_pretrained(CKPT_DIR)
print(f"Checkpoint guardado en {CKPT_DIR}")

Checkpoint guardado en /content/drive/MyDrive/MASTER/Tercer_semestre/NLP_2/Competencia/decoder_checkpoint


## 5 · Inferencia + Evaluación

In [11]:
def normalize_tool_call(raw: str) -> str:
    raw = raw.strip()
    if raw.lower().startswith("no_action"):
        return "no_action"

    m = re.match(r"^(\w+)\s*\((.*)\)$", raw, re.DOTALL)
    if not m:
        m = re.search(r"(\w+)\s*\(([^)]*)\)", raw)
        if not m:
            return "no_action"

    name, params_str = m.group(1).strip(), m.group(2).strip()

    if name not in VALID_TOOLS:
        candidates = get_close_matches(name, VALID_TOOLS, n=1, cutoff=0.6)
        name = candidates[0] if candidates else "no_action"

    if name == "no_action":
        return "no_action"

    parsed = {k: v for k, v in re.findall(r"(\w+)\s*=\s*('[^']*'|\d+)", params_str)}
    expected = list(TOOLS_DEF[name].get("parameters", {}).keys())
    parts = [f"{k}={parsed[k]}" for k in expected if k in parsed]

    return f"{name}({','.join(parts)})" if parts or not expected else "no_action"


model.eval()
MAX_NEW_TOKS = 80

def predict(query: str, ctx_chunks: list) -> str:
    prompt = make_prompt(query, ctx_chunks[:PROMPT_K])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKS,
            do_sample=False,
            repetition_penalty=1.05,
        )

    raw = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return normalize_tool_call(raw)

In [12]:
# Evaluar en validation set
print(f"Evaluando {len(val_df)} ejemplos...")
val_preds = [predict(row["query"], row["context_chunks"]) for _, row in val_df.iterrows()]
val_gts   = val_df["tool_call"].tolist()

exact_match = sum(p == g for p, g in zip(val_preds, val_gts)) / len(val_gts)
print(f"\nExact Match (validation): {exact_match*100:.2f}%")

# Análisis de errores
def extract_tool_name(tc): return tc.split("(")[0].strip()

errors = [
    {"query": q, "gt": g, "pred": p, "tool": extract_tool_name(g)}
    for q, g, p in zip(val_df["query"], val_gts, val_preds) if p != g
]
print(f"Errores: {len(errors)}")
if errors:
    err_df = pd.DataFrame(errors)
    print("\nErrores por herramienta:")
    print(err_df["tool"].value_counts().to_string())
    print("\nPrimeros 3 errores:")
    for e in errors[:3]:
        print(f"  Q:    {e['query'][:70]}")
        print(f"  GT:   {e['gt']}")
        print(f"  Pred: {e['pred']}\n")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Evaluando 257 ejemplos...


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin


Exact Match (validation): 92.22%
Errores: 20

Errores por herramienta:
tool
activate_protocol       10
send_alert               4
send_message             2
control_system           1
schedule_maintenance     1
get_telemetry            1
calculate_trajectory     1

Primeros 3 errores:
  Q:    Tucán radiation sensors are at 8.6 mSv/hr and rising—what emergency re
  GT:   activate_protocol(protocol_id='MASA-SEC-004',scope='station_wide')
  Pred: send_alert(module='tucan',severity='critical',reason='radiation_spike')

  Q:    Jaguar oxygen concentration has dropped to 11.2% — Engineer Kozlov rep
  GT:   activate_protocol(protocol_id='MASA-SEC-002',scope='station_wide')
  Pred: send_alert(module='jaguar',severity='critical',reason='oxygen_leak')

  Q:    Kai just reported unusual fluctuations in Quetzal's experimental react
  GT:   send_message(recipient='specialist_1',priority='high')
  Pred: send_message(recipient='specialist_1',priority='urgent')



## 6 · Generar submission.csv

In [13]:
print(f"Prediciendo {len(test_df)} queries de test...")
test_preds = []
for i, row in test_df.iterrows():
    test_preds.append(predict(row["query"], row["context_chunks"]))
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(test_df)}...")

# Validar formato
invalid = [
    (i, p) for i, p in enumerate(test_preds)
    if p != "no_action" and (not re.match(r"^(\w+)\(.*\)$", p) or p.split("(")[0] not in VALID_TOOLS)
]
if invalid:
    print(f"\nADVERTENCIA: {len(invalid)} predicciones con formato inválido")
else:
    print(f"\nOK — todas las {len(test_preds)} predicciones son válidas.")

# Guardar
submission = pd.DataFrame({"id": test_df["id"].tolist(), "tool_call": test_preds})
submission.to_csv(BASE / "submission.csv", index=False)
print("submission.csv guardado.")
submission.head(10)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Prediciendo 766 queries de test...


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

  100/766...


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

...

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

  700/766...


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin


OK — todas las 766 predicciones son válidas.
submission.csv guardado.


,id,tool_call
0,Q-00794,"request_supply(category='scientific',urgency='..."
1,Q-00365,"activate_protocol(protocol_id='MASA-SEC-008',s..."
2,Q-01558,"get_module_status(module='tucan',system='life_..."
3,Q-03295,"activate_protocol(protocol_id='MASA-SEC-005',s..."
4,Q-00491,"control_system(module='jaguar',system='heating..."
5,Q-00149,"send_message(recipient='all_crew',priority='me..."
6,Q-01229,"activate_protocol(protocol_id='MASA-SEC-001',s..."
7,Q-03457,"schedule_maintenance(module='quetzal',task='se..."
8,Q-03300,"activate_protocol(protocol_id='MASA-SEC-011',s..."
9,Q-03530,no_action
